## Data Transformation Plan

### Key Metrics
- Views
- Likes
- Duration
- Tags

### Data Issues to Fix
- Remove duplicates
- Fix wrong data types
- Handle missing or empty fields

### Derived Features
- Engagement Rate = (likes / views) * 100
- Duration in Minutes = duration / 60
- Views per Day = views / days_since_published
- Tag Count = number of tags


In [13]:
import util
import importlib
importlib.reload(util)

<module 'util' from '/Users/anaghanair/Downloads/youtube-pipeline/util.py'>

In [14]:
import pandas as pd
df = pd.read_csv("data/youtube_data.csv")

In [15]:
df.columns

Index(['video_id', 'title', 'published_at', 'channel_title', 'view_count',
       'like_count', 'dislike_count', 'comment_count', 'duration', 'tags'],
      dtype='object')

In [16]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   video_id       100 non-null    object
 1   title          100 non-null    object
 2   published_at   100 non-null    object
 3   channel_title  100 non-null    object
 4   view_count     100 non-null    int64 
 5   like_count     100 non-null    int64 
 6   dislike_count  100 non-null    int64 
 7   comment_count  100 non-null    int64 
 8   duration       100 non-null    object
 9   tags           100 non-null    object
dtypes: int64(4), object(6)
memory usage: 7.9+ KB


In [17]:
from util import duration_parse

df["duration_parsed"] = df["duration"].apply(duration_parse).round(3)

In [18]:
df["published_at"] = pd.to_datetime(df["published_at"],
                                    errors='coerce',
                                    yearfirst=True
                                    )

df["hour"] = df["published_at"].dt.hour


from util import which_bucket

df["time_bucket"] = which_bucket(df["hour"])
df.head()

,video_id,title,published_at,channel_title,view_count,like_count,dislike_count,comment_count,duration,tags,duration_parsed,hour,time_bucket
0,SixTbTc6YCg,"ice spice, tokischa - thootie",2025-12-05 17:00:06+00:00,IceSpiceVEVO,516179,48825,0,2663,PT2M33S,"['Ice Spice', 'Tokischa', '10K Projects/Capito...",2.550,17,Afternoon
1,VI6XO8EijT4,digital circus ep 7 trailer!!,2025-12-05 20:00:28+00:00,GLITCH,4246050,432146,0,25506,PT58S,"['glitch', 'glitch productions', 'digital circ...",0.967,20,Evening
2,eWPZLDvRebU,5000 days - [hardcore minecraft],2025-12-05 23:00:54+00:00,Luke TheNotable,635604,47461,0,5484,PT2H1S,"['luke thenotable', 'Minecraft', '100 Days', '...",120.017,23,Night
3,FeaRXx2VDhI,top 50 christmas songs of all time 🎄 best chri...,2025-12-05 11:30:23+00:00,Christmas Songs and Carols - Love to Sing,563186,2185,0,122,PT2H19M11S,"['top christmas song', 'christmas songs', 'top...",139.183,11,Morning
4,IaEtA56pd_w,iron lung: final trailer,2025-12-05 20:48:18+00:00,Markiplier,2084361,325040,0,16823,PT1M45S,"['markiplier', 'iron lung', 'iron lung movie',...",1.750,20,Evening


In [19]:
df["time_bucket"].value_counts()

time_bucket
Late Night    26
Afternoon     24
Night         23
Evening       15
Morning       12
Name: count, dtype: int64

In [20]:
df["is_weekend"] = df["published_at"].dt.dayofweek.isin([5,6]).astype(int)
df["is_weekend"].value_counts()

is_weekend
0    76
1    24
Name: count, dtype: int64

In [21]:
import ast
import numpy as np

df["engagement_rate"] = (df["like_count"] / df["view_count"]) * 100
days_since_pub = (pd.Timestamp.utcnow() - df["published_at"]).dt.days.clip(lower=1)
df["views_per_day"] = df["view_count"] / days_since_pub


df["tags"] = df["tags"].replace('NaN',np.nan).apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
df["tag_count"] = df["tags"].apply(len) 


In [22]:
df.drop(columns=["hour","dislike_count","duration"],inplace=True)

In [23]:
df.to_csv("/Users/anaghanair/Downloads/youtube-pipeline/data/youtube_clean.csv",index=False) 

In [24]:
df.head()

,video_id,title,published_at,channel_title,view_count,like_count,comment_count,tags,duration_parsed,time_bucket,is_weekend,engagement_rate,views_per_day,tag_count
0,SixTbTc6YCg,"ice spice, tokischa - thootie",2025-12-05 17:00:06+00:00,IceSpiceVEVO,516179,48825,2663,"[Ice Spice, Tokischa, 10K Projects/Capitol Rec...",2.550,Afternoon,0,9.458928,516179.0,4
1,VI6XO8EijT4,digital circus ep 7 trailer!!,2025-12-05 20:00:28+00:00,GLITCH,4246050,432146,25506,"[glitch, glitch productions, digital circus, t...",0.967,Evening,0,10.177600,4246050.0,19
2,eWPZLDvRebU,5000 days - [hardcore minecraft],2025-12-05 23:00:54+00:00,Luke TheNotable,635604,47461,5484,"[luke thenotable, Minecraft, 100 Days, minecra...",120.017,Night,0,7.467071,635604.0,19
3,FeaRXx2VDhI,top 50 christmas songs of all time 🎄 best chri...,2025-12-05 11:30:23+00:00,Christmas Songs and Carols - Love to Sing,563186,2185,122,"[top christmas song, christmas songs, top chri...",139.183,Morning,0,0.387971,563186.0,14
4,IaEtA56pd_w,iron lung: final trailer,2025-12-05 20:48:18+00:00,Markiplier,2084361,325040,16823,"[markiplier, iron lung, iron lung movie, offic...",1.750,Evening,0,15.594228,2084361.0,11
